![Week 3 Day 5](../../assets/banners/w3d5.png)

# Week 3 Day 5 - Solution: Three Attempts, Three Different Mistakes

The exercise: a stand-in that gets the paper total wrong, then forgets a methodology, then succeeds. Predict, stream, read the feedback, approve.

⏮ Back to the lab: `week3/day5.ipynb`

In [ ]:
from langchain_core.runnables import RunnableLambda
from langgraph.types import Command

from gradewise.config import DATA_DIR, OUTPUT_DIR
from gradewise.db import load_exam
from gradewise.exam.graph import build_exam_graph, start_input
from gradewise.schemas import ExamPaper

EXAM_DIR = DATA_DIR / "exams" / "cs_midterm"
DEMO_DB = OUTPUT_DIR / "week3_day5_demo.db"


class FakeStructuredLLM:
    """🎭 STAND-IN model: returns canned ExamPapers in order and records every prompt."""

    def __init__(self, outputs: list) -> None:
        self.outputs, self.calls = list(outputs), []

    def with_structured_output(self, schema, **kwargs):
        def respond(messages):
            self.calls.append(messages)
            out = self.outputs.pop(0)
            return out if isinstance(out, schema) else schema.model_validate(out)

        return RunnableLambda(respond)


expected = ExamPaper.model_validate_json((EXAM_DIR / "expected_exam.json").read_text(encoding="utf-8"))

wrong_total = expected.model_copy(update={"total_marks": 60.0})
no_methodology = expected.model_copy(deep=True)
next(q for q in no_methodology.questions if q.id == "Q5").methodology = ""

print("Attempt 1 problems:", wrong_total.problems())
print("Attempt 2 problems:", no_methodology.problems())

**Prediction:** attempt 1 fails validation (total), attempt 2 fails (methodology), attempt 3 is clean. With `max_attempts=3` that's exactly enough: `extract` runs 3 times, `retry` twice, then the graph pauses at `admin_review` with **no** problems.

In [ ]:
llm = FakeStructuredLLM([wrong_total, no_methodology, expected])
graph = build_exam_graph(llm=llm, db=DEMO_DB, write_json=False)
config = {"configurable": {"thread_id": "day5-solution"}}

for update in graph.stream(start_input("cs_midterm", max_attempts=3), config, stream_mode="updates"):
    for node, change in update.items():
        print(f"▸ {node:13} {'⏸ paused for Asha' if node == '__interrupt__' else sorted(change or {})}")

In [ ]:
for n, messages in enumerate(llm.calls, 1):
    feedback = messages[-1].content if len(messages) > 2 else "(first attempt: no feedback)"
    print(f"── attempt {n} was told ──\n{feedback.split(chr(10) + chr(10))[0]}\n")

In [ ]:
final = graph.invoke(Command(resume={"approved": True}), config)
print("Stored?", final["stored"], "· attempts:", final["attempts"])
assert load_exam("cs_midterm", db=DEMO_DB) == expected